In [1]:
import pandas as pd
import os

def load_all_sheets(file_path):
    """读取Excel所有Sheet并合并，增加Sheet名作为来源列"""
    if not os.path.exists(file_path):
        return None
    xls = pd.ExcelFile(file_path)
    all_data = []
    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name)
        df['Source_Sheet'] = sheet_name
        all_data.append(df)
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

def compare_reports(old_somatic, new_somatic, old_filtered, new_filtered):
    print("="*60)
    print(" SakitNeo Report Iteration Audit Report ")
    print("="*60)

    # 1. 格式/列名变化检查
    old_df = load_all_sheets(old_somatic)
    new_df = load_all_sheets(new_somatic)
    
    old_cols = set(old_df.columns)
    new_cols = set(new_df.columns)
    
    print(f"\n[1] Schema Changes (Columns):")
    added_cols = new_cols - old_cols
    removed_cols = old_cols - new_cols
    print(f"    - Added columns: {added_cols if added_cols else 'None'}")
    print(f"    - Removed columns: {removed_cols if removed_cols else 'None'}")

    # 2. 宏观数量统计 (Summary)
    print(f"\n[2] Global Distribution Summary:")
    
    def get_summary(s_file, f_file):
        s = load_all_sheets(s_file)
        f = load_all_sheets(f_file)
        combined = pd.concat([s, f], ignore_index=True)
        return combined['Primary_Status'].value_counts().to_dict(), combined

    old_counts, old_full = get_summary(old_somatic, old_filtered)
    new_counts, new_full = get_summary(new_somatic, new_filtered)

    stats = pd.DataFrame({'Old_Version': old_counts, 'New_Version': new_counts}).fillna(0).astype(int)
    stats['Diff'] = stats['New_Version'] - stats['Old_Version']
    print(stats)

    # 3. 关键状态漂移分析 (Status Transitions)
    # 通过 VariantKey 进行关联
    merged = pd.merge(
        old_full[['VariantKey', 'Primary_Status', 'Gene', 'HGVSp']], 
        new_full[['VariantKey', 'Primary_Status', 'Whitelist_Driver', 'Biological_Aware']], 
        on='VariantKey', suffixes=('_Old', '_New')
    )

    print(f"\n[3] Significant Status Transitions:")
    
    # A. 黑名单拦截效果
    blacklist_count = len(merged[merged['Primary_Status_New'] == 'Blacklist_Noise'])
    print(f"    - Variants moved to Blacklist: {blacklist_count}")
    
    # B. 白名单捞回效果 (重要！)
    rescued = merged[(merged['Primary_Status_Old'] == 'Noise') & (merged['Primary_Status_New'] == 'Somatic')]
    print(f"    - Whitelist Rescued (Noise -> Somatic): {len(rescued)}")
    for _, row in rescued.iterrows():
        print(f"      * [RESCUED] {row['Gene']} {row['HGVSp']}")

    # C. 丢弃检查 (不该发生的丢失)
    lost = merged[(merged['Primary_Status_Old'] == 'Somatic') & (merged['Primary_Status_New'] == 'Noise')]
    if not lost.empty:
        print(f"    - WARNING: Somatic variants demoted to Noise: {len(lost)}")
        print(lost[['VariantKey', 'Gene']])
    else:
        print("    - Consistency Check: No Somatic variants were lost. (Pass)")

    # 4. Aware 标签覆盖率
    aware_count = len(new_full[new_full['Biological_Aware'] != '/'])
    print(f"\n[4] Clinical Awareness Flags:")
    print(f"    - Total variants flagged with Awareness: {aware_count}")
    if aware_count > 0:
        print(new_full[new_full['Biological_Aware'] != '/']['Biological_Aware'].value_counts())

    print("\n" + "="*60)

if __name__ == "__main__":
    # 请根据实际路径修改以下四个文件名
    compare_reports(
        "/home/jovyan/work/09.data_CCS_sdfyy/05.Result.Sakit2Neo/CS019_2025/reports/CS019_2025.Somatic_NeoPeptides.xlsx", # 旧版Somatic
        "/home/jovyan/work/09.data_CCS_sdfyy/05.Result.Sakit2Neo/CS019_2025/reports/CS019_2025.Somatic_NeoPeptides_v2.xlsx",     # 新版Somatic
        "/home/jovyan/work/09.data_CCS_sdfyy/05.Result.Sakit2Neo/CS019_2025/reports/CS019_2025.Filtered_variants.xlsx",  # 旧版Filtered
        "/home/jovyan/work/09.data_CCS_sdfyy/05.Result.Sakit2Neo/CS019_2025/reports/CS019_2025.Filtered_variants_v2.xlsx"       # 新版Filtered
    )

 SakitNeo Report Iteration Audit Report 

[1] Schema Changes (Columns):
    - Added columns: {'Sample Neoantigen Quality & Warning Report', 'Unnamed: 1', 'Whitelist_Driver', 'Biological_Aware'}
    - Removed columns: None

[2] Global Distribution Summary:
                 Old_Version  New_Version  Diff
Germline               54361        54361     0
Noise                  24415        23346 -1069
Somatic                 1212         1124   -88
LOH                      291          291     0
Blacklist_Noise            0         1157  1157

[3] Significant Status Transitions:
    - Variants moved to Blacklist: 1157
    - Whitelist Rescued (Noise -> Somatic): 0
    - Consistency Check: No Somatic variants were lost. (Pass)

[4] Clinical Awareness Flags:
    - Total variants flagged with Awareness: 97
Biological_Aware
APM_Loss              47
Polymerase_Defect     19
MMR_Defect (MSI-H)    14
IFNg_Pathway_Loss      7
Splice_Altered         3
Name: count, dtype: int64

